# Decision Science Lab: Causal Analytics for Retail Promotions

**Business question:** Should the retailer continue, stop, or target promotions?

This first notebook establishes the baseline descriptive analysis and shows why a naive promoted-vs-non-promoted comparison can be misleading.

## 1. Load Data

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("default")

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "retail_promotions_synthetic.csv"

# Support running from either the repo root or the notebooks directory.
if not DATA_PATH.exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_PATH = PROJECT_ROOT / "data" / "retail_promotions_synthetic.csv"

FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

In [ ]:
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.columns.tolist()

## 2. Sanity Checks

In [ ]:
promotion_rate = df["promotion_flag"].mean()
print(f"Promotion rate: {promotion_rate:.3f}")

In [ ]:
promotion_distribution = pd.DataFrame(
    {
        "row_count": df["promotion_flag"].value_counts().sort_index(),
        "share": df["promotion_flag"].value_counts(normalize=True).sort_index(),
    }
)
promotion_distribution.index = ["non_promoted", "promoted"]
promotion_distribution

In [ ]:
outcome_cols = ["units_sold", "revenue", "gross_profit", "net_profit"]
df[outcome_cols].describe().T.round(2)

In [ ]:
missing_values = df.isna().sum().to_frame("missing_count")
missing_values["missing_rate"] = missing_values["missing_count"] / len(df)
missing_values.query("missing_count > 0")

## 3. Naive Comparison

In [ ]:
naive_group_averages = df.groupby("promotion_flag")[outcome_cols].mean().round(2)
naive_group_averages.index = ["non_promoted", "promoted"]
naive_group_averages

In [ ]:
naive_net_profit_diff = (
    df.loc[df["promotion_flag"].eq(1), "net_profit"].mean()
    - df.loc[df["promotion_flag"].eq(0), "net_profit"].mean()
)

print(f"Naive net_profit difference: {naive_net_profit_diff:.2f}")

The naive comparison treats promoted and non-promoted rows as if they were exchangeable. In this synthetic data, that assumption is intentionally false: promotions are targeted toward rows with stronger expected demand.

## 4. Ground Truth Comparison

In [ ]:
true_incremental_profit_promoted = df.loc[
    df["promotion_flag"].eq(1), "incremental_profit_true"
].mean()

bias = naive_net_profit_diff - true_incremental_profit_promoted

ground_truth_comparison = pd.DataFrame(
    {
        "metric": [
            "naive net_profit difference",
            "true incremental profit among promoted rows",
            "bias: naive minus true",
        ],
        "value": [naive_net_profit_diff, true_incremental_profit_promoted, bias],
    }
)

ground_truth_comparison["value"] = ground_truth_comparison["value"].round(2)
ground_truth_comparison

The ground truth column exists only because this is synthetic data. In a real retail setting, the analyst would observe sales and profit, but not the counterfactual profit that would have occurred without the promotion.

## 5. Visualizations

In [ ]:
avg_net_profit = df.groupby("promotion_flag")["net_profit"].mean().reindex([0, 1])

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Non-promoted", "Promoted"], avg_net_profit, color=["#6B7280", "#2563EB"])
ax.set_title("Average Net Profit by Promotion Status")
ax.set_ylabel("Average net profit")
ax.spines[["top", "right"]].set_visible(False)

figure_path = FIGURE_DIR / "average_net_profit_by_promotion_flag.png"
fig.tight_layout()
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {figure_path}")

In [ ]:
estimate_values = pd.Series(
    {
        "Naive\ncomparison": naive_net_profit_diff,
        "True\nincremental profit": true_incremental_profit_promoted,
    }
)

fig, ax = plt.subplots(figsize=(6, 4))
colors = ["#2563EB" if value >= 0 else "#DC2626" for value in estimate_values]
ax.bar(estimate_values.index, estimate_values.values, color=colors)
ax.axhline(0, color="#111827", linewidth=1)
ax.set_title("Naive Estimate vs True Effect")
ax.set_ylabel("Net profit effect")
ax.spines[["top", "right"]].set_visible(False)

figure_path = FIGURE_DIR / "naive_estimate_vs_true_effect.png"
fig.tight_layout()
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {figure_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0, 1, 26)

for flag, label, color in [
    (0, "Non-promoted", "#6B7280"),
    (1, "Promoted", "#2563EB"),
]:
    ax.hist(
        df.loc[df["promotion_flag"].eq(flag), "treatment_probability"],
        bins=bins,
        alpha=0.60,
        density=True,
        label=label,
        color=color,
    )

ax.set_title("Treatment Probability Distribution by Promotion Status")
ax.set_xlabel("Treatment probability")
ax.set_ylabel("Density")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

figure_path = FIGURE_DIR / "treatment_probability_distribution_by_promotion_flag.png"
fig.tight_layout()
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {figure_path}")

## 6. Interpretation

The naive analysis is misleading because promotion assignment is not random. Promotions are more common in store-product-week rows that already have stronger demand signals: higher baseline demand stores, more popular products, holiday periods, seasonal periods, and strategic categories.

Selection bias appears when the promoted group has higher expected performance before the promotion effect is added. As a result, the promoted rows can show higher observed units sold, revenue, or net profit even when the promotion itself does not create that much incremental value.

In this dataset, promoted rows can look better because managers target promotions where sales were already likely to be high. The true synthetic treatment effect shows the counterfactual business impact: some promotions generate incremental profit, but others mainly discount demand that would have happened anyway or add promotion costs that outweigh the extra units sold.

This is why causal methods are needed next. The next analysis should adjust for confounding before making a decision about whether the retailer should continue, stop, or target promotions.